In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from statsmodels.tsa.seasonal import STL

input_path = "../dataset/quantity_weekly_decor_outlier_fixed.csv"
output_dir = "../decomposition_stl_comparison"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

df = pd.read_csv(input_path)
weeks = df.columns[1:]
dates = pd.date_range(start='2010-12-01', periods=len(weeks), freq='W')
p = 6

for index, row in df.iterrows():
    stock_code = str(row['StockCode'])
    series_data = row[1:].values.astype(float)
    ts_org = pd.Series(series_data, index=dates)
    
    # Chuỗi vi phân
    ts_diff = ts_org.diff().dropna()
    
    try:
        # Phân rã cả hai chuỗi
        res_org = STL(ts_org, period=p, robust=True).fit()
        res_diff = STL(ts_diff, period=p, robust=True).fit()
        
        fig, axes = plt.subplots(4, 2, figsize=(16, 12), sharex='col')
        
        axes[0, 0].plot(res_org.observed, color='blue', linewidth=1.2)
        axes[0, 0].set_title(f'{stock_code} - Chuỗi gốc')
        
        axes[1, 0].plot(res_org.trend, color='blue', linewidth=1.2)
        axes[1, 0].set_title('Xu hướng')
        
        axes[2, 0].plot(res_org.seasonal, color='blue', linewidth=1.2)
        axes[2, 0].set_title('Mùa vụ')
        
        axes[3, 0].plot(res_org.resid, color='blue', linewidth=1.2)
        axes[3, 0].set_title('Nhiễu')
        axes[3, 0].axhline(0, color='red', linestyle='--', linewidth=0.8)

        axes[0, 1].plot(res_diff.observed, color='blue', linewidth=1.2)
        axes[0, 1].set_title(f'{stock_code} - Vi phân bậc 1')
        
        axes[1, 1].plot(res_diff.trend, color='blue', linewidth=1.2)
        axes[1, 1].set_title('Xu hướng')
        
        axes[2, 1].plot(res_diff.seasonal, color='blue', linewidth=1.2)
        axes[2, 1].set_title('Mùa vụ')
        
        axes[3, 1].plot(res_diff.resid, color='blue', linewidth=1.2)
        axes[3, 1].set_title('Nhiễu')
        axes[3, 1].axhline(0, color='red', linestyle='--', linewidth=0.8)

        for ax in axes.flat:
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.tick_params(axis='both', which='major', labelsize=8)

        plt.tight_layout()
        plt.savefig(f"{output_dir}/{stock_code}.png", dpi=150)
        plt.close(fig)
        
    except Exception as e:
        print(f"{stock_code}: {e}")